# Probe Guidance: Descriptive Statistics for the LL Probe Guidance Dataset

In [1]:
import sys
sys.path.insert(0, "/home/jack/code/vjepa2-probe-guidance/vjepa2")
print(sys.path)

['/home/jack/code/vjepa2-probe-guidance/vjepa2', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages', '/home/jack/code/vjepa2-probe-guidance/vjepa2/src', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/rerun_sdk']


In [2]:
import numpy as np
import torch
from tqdm import tqdm
from scipy.spatial.transform import Rotation
import matplotlib.pyplot as plt

from app.vjepa_ll_probe_guidance.ll_probe_guidance import LLProbeGuidanceDataset
from app.vjepa_ll_probe_guidance.transforms import make_transforms

In [3]:
def load_clips(sample, device):
    clips = sample[0].to(device, non_blocking=True)  # [B C T H W]
    actions = sample[1].to(device, non_blocking=True)  # [B T-1 6]
    states = sample[2].to(device, non_blocking=True)  # [B T 6]
    extrinsics = sample[3].to(device, non_blocking=True)  # [B T 6]
    return (clips, actions, states, extrinsics)

In [4]:
crop_size = 256
tokens_per_frame = 256
clip_size = 8
fps = 4

transform = make_transforms(
    crop_size=crop_size,
)

train_dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/train",
    frames_per_clip=clip_size,
    frame_skip=1,
    frames_per_second=fps,
    transform=transform,
    is_train=False,
)

loader = torch.utils.data.DataLoader(
    train_dataset,
    shuffle=False,
    batch_size=128,
    num_workers=32,
)

Scanning 60 episodes for valid tracking clips...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:01<00:00, 43.90it/s]

Retained 60 episodes.


In [12]:
delta_translations = []
delta_rotations = []

for _, actions_batch, _, _, _ in tqdm(loader, total=len(loader)):
    actions_batch_np = actions_batch.numpy()

    individual_delta_translations = [sample[:, :3] for sample in actions_batch_np]
    individual_delta_rotations = [sample[:, 3:] for sample in actions_batch_np]

    delta_translations.extend(individual_delta_translations)
    delta_rotations.extend(individual_delta_rotations)   

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 950/950 [08:16<00:00,  1.91it/s]


## Clip Level Distance Stats and Visualizations
What makes a clip "fine-grained" is somewhat arbitrary and also task specific. But for DVT, a clip that involves fine-grained movements will likely have short total and net translational distances, say 3cm. Rotation is also very important to consider, but it's less clear what the threshold should be for it. You may need to rotate a somewhat substantial amount even when you are on the vein position to get a better quality view of the vein. However, there is certainly a reasonably low threshold for fine-grained rotation guidance. I have randomly chosen 10 degrees as a threshold, which may be too low (or too high even?)

Keep in mind that total and net distances are highly dependent on the chosen clip size and FPS. higher clip sizes and lower FPS values will increase total distances (and most likely net distances as well). As of writing this, the VJEPA2 AC Predictor for the probe guidance dataset was trained on clips of size 8 and sampled at 4FPS, which means these clips are 2 seconds long and it's reasonable to expect a fair amount of probe movement to happen within 2 seconds. 

Descriptive statistics of distance can help inform what are appropriate values for clip size and FPS for the VJEPA2 AC Predictor to train on. However, it's not entirely clear to me yet what impact adjusting these parameters will have on the model. It may be possible that training on more fine-grained clips will lead to worse performance.

In [ ]:
total_clip_translations_m = []
net_clip_translations_m = []

total_clip_angular_motions_deg = []
net_clip_angular_motions_deg = []

for d_trans, d_rpy in tqdm(zip(delta_translations, delta_rotations), total=len(delta_translations)):
    step_distances = np.linalg.norm(d_trans, axis=-1)
    total_path_distance = np.sum(step_distances)

    delta_rots = Rotation.from_euler('xyz', d_rpy, degrees=True)
    step_rot_angles_rad = np.linalg.norm(delta_rots.as_rotvec(), axis=-1)
    total_angular_distance_deg = np.degrees(np.sum(step_rot_angles_rad))

    current_rot = Rotation.identity()
    current_pos = np.zeros(3)

    for i in range(len(d_trans)):
        current_pos += current_rot.apply(d_trans[i])
        current_rot = current_rot * delta_rots[i]

    net_displacement = np.linalg.norm(current_pos)
    net_rotation_angle_deg = np.degrees(np.linalg.norm(current_rot.as_rotvec()))

    total_clip_translations_m.append(total_path_distance)
    net_clip_translations_m.append(net_displacement)

    total_clip_angular_motions_deg.append(total_angular_distance_deg)
    net_clip_angular_motions_deg.append(net_rotation_angle_deg)

total_clip_translations_m = np.array(total_clip_translations_m)
net_clip_translations_m = np.array(net_clip_translations_m)

total_clip_angular_motions_deg = np.array(total_clip_angular_motions_deg)
net_clip_angular_motions_deg = np.array(net_clip_angular_motions_deg)

### Clip Level Distance Histograms
Histograms can give us a good idea of how many clips fall under the category of being "fine-grained". If not many fall under that category, then the chosen clip size and FPS might be inadequate.

In [ ]:
n_bins = 25

fig, axs = plt.subplots(1, 2, sharey=True, tight_layout=True)

x1 = total_clip_translations_m
x2 = net_clip_translations_m

axs[0].hist(x1[x1 < 0.03], bins=n_bins)
axs[1].hist(x2[x2 < 0.03], bins=n_bins)

fig.suptitle("Total and Net Clip Translational Distances < 3cm")
fig.supylabel("Clip/Sample Count")
fig.supxlabel("Meters")

plt.show()

In [ ]:
n_bins = 25

fig, axs = plt.subplots(1, 2, sharey=True, tight_layout=True)

x1 = total_clip_angular_motions_deg
x2 = net_clip_angular_motions_deg

axs[0].hist(x1[x1 < 10], bins=n_bins)
axs[1].hist(x2[x2 < 10], bins=n_bins)

fig.suptitle("Total and Net Clip Rotational Distances < 10deg")
fig.supylabel("Clip/Sample Count")
fig.supxlabel("Degrees")

plt.show()

## Frame-to-Frame Level Stats and Visualizations
Looking at the frame-to-frame level actions/distances is also important, if not moreso, than looking at the clip level distances. Given how the teacher forcing and rollout components of the training work, the frame-to-frame actions are what the predictor model is actually being trained on. So if the model is never actually seeing fine-grained actions, then it's reasonable to assume that it will not learn the correct movement patterns necessary to give fine-grained probe guidance.

The FPS of the clips has the biggest impact on the frame-to-frame level, since that is what directly controls the time difference between each frame (and also the physical distance in probe position). More time between frames probably has a positive linearish relationship with distance (as time increases between frames, so does distance). Clip size probably does not have much of an impact, besides potentially introducing less data (this is unintuitive, but higher clip sizes will likely cause more clip candidates to be dropped due to missing tracking data).

In [ ]:
all_actions_translations = [action for clip_actions in delta_translations for action in clip_actions]
all_actions_rotations = [action for clip_actions in delta_rotations for action in clip_actions]

action_translation_distances = []
action_angular_motions = []
for translation, rotation in tqdm(zip(all_actions_translations, all_actions_rotations), total=len(all_actions_translations)):
    action_translation_distances.append(np.linalg.norm(translation))

    rotation = Rotation.from_euler('xyz', rotation, degrees=True)
    angular_motion = np.linalg.norm(rotation.as_rotvec(), axis=-1)
    action_angular_motions.append(np.degrees(angular_motion))

action_translation_distances = np.array(action_translation_distances)
action_angular_motions = np.array(action_angular_motions)

### Frame-to-Frame Level Distance Histograms
The clip level and the frame-to-frame level histograms can paint quite different pictures. At the clip level, it may appear that the samples are inadequate for fine-grained guidance. While at the frame-to-frame level, we could find that many actions within the clips are actually fine-grained. As mentioned previously, we need to find a sweet spot for actions such that the AC predictor will be able to see enough reasonably fine-grained motions to understand how fine-grained actions will change/affect the environment.

In [ ]:
n_bins = 100

fig, ax = plt.subplots(1, 1, tight_layout=True)

x1 = action_translation_distances

ax.hist(x1[x1 < 0.1], bins=n_bins)

fig.suptitle("Frame-to-Frame Action Translational Distances < 1cm")
fig.supylabel("Action Count")
fig.supxlabel("Meters")

plt.show()

In [ ]:
n_bins = 100

fig, ax = plt.subplots(1, 1, tight_layout=True)

x1 = action_angular_motions

ax.hist(x1[x1 < 10], bins=n_bins)

fig.suptitle("Frame-to-Frame Action Angular Motions < 10deg")
fig.supylabel("Action Count")
fig.supxlabel("Degrees")

plt.show()

In [9]:
crop_size = 256
tokens_per_frame = 256
clip_size = 8
fps = 4

transform = make_transforms(
    crop_size=crop_size,
)

train_dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/train",
    frames_per_clip=clip_size,
    frame_skip=1,
    frames_per_second=fps,
    transform=transform,
    is_train=False,
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    shuffle=False,
    batch_size=1,
    num_workers=32,
)

print(f"{len(train_loader)=}")

Scanning 60 episodes for valid tracking clips...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:01<00:00, 49.37it/s]

Retained 60 episodes.
len(train_loader)=121548


In [10]:
crop_size = 256
tokens_per_frame = 256
clip_size = 8
fps = 4

transform = make_transforms(
    crop_size=crop_size,
)

val_dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/val",
    frames_per_clip=clip_size,
    frame_skip=1,
    frames_per_second=fps,
    transform=transform,
    is_train=False,
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    shuffle=False,
    batch_size=1,
    num_workers=32,
)

print(f"{len(val_loader)=}")

Scanning 8 episodes for valid tracking clips...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 42.32it/s]

Retained 8 episodes.
len(val_loader)=17695


In [11]:
crop_size = 256
tokens_per_frame = 256
clip_size = 8
fps = 4

transform = make_transforms(
    crop_size=crop_size,
)

test_dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/test",
    frames_per_clip=clip_size,
    frame_skip=1,
    frames_per_second=fps,
    transform=transform,
    is_train=False,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    shuffle=False,
    batch_size=1,
    num_workers=32,
)

print(f"{len(test_loader)=}")

Scanning 7 episodes for valid tracking clips...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 39.68it/s]

Retained 7 episodes.
len(test_loader)=17061
